In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1xCaVkRzc7bSATDO6aosTqAQABqxQmp7f", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


# Notebook 3: Load Balancing and the 5D Parallelism Grid -- Vizuara

In this notebook, we will tackle the most critical challenge in MoE training: **load balancing**. We will implement the auxiliary loss that prevents expert collapse, build the capacity-based token dropping mechanism, and finally explore how expert parallelism fits into the complete **5D parallelism grid**.

**What you will build:**
- The auxiliary load balancing loss from the Switch Transformer paper
- Expert capacity buffers with token overflow/dropping
- A MoE layer that trains with load balancing
- A 5D parallelism configuration calculator

**Estimated time:** 40--60 minutes

**Prerequisites:** Notebooks 1 and 2

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import sys

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs fine on CPU.")

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Imbalance Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_imbalance_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. The Load Imbalance Problem

Without any intervention, the router tends to collapse: it sends most tokens to a small number of "popular" experts. This is the **rich-get-richer** dynamic:

1. Expert 3 produces slightly better outputs early in training
2. Router sends more tokens to Expert 3
3. Expert 3 gets more gradient updates, improves further
4. Router sends even more tokens to Expert 3
5. Other experts starve -- **expert collapse**

Let us simulate this phenomenon.

In [ ]:
#@title 🎧 Code Walkthrough: Simulate Collapse Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_simulate_collapse_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.w2(self.act(self.w1(x)))


class MoELayerNoBalance(nn.Module):
    """MoE layer WITHOUT load balancing -- to show the collapse."""
    def __init__(self, d_model, d_ff, num_experts, top_k=1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, S, d = x.shape
        logits = self.router(x)
        probs = F.softmax(logits, dim=-1)
        top_k_probs, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(x)
        for k in range(self.top_k):
            for e in range(self.num_experts):
                mask = (top_k_idx[:, :, k] == e)
                if mask.any():
                    tokens = x[mask]
                    expert_out = self.experts[e](tokens)
                    output[mask] += top_k_weights[:, :, k][mask].unsqueeze(-1) * expert_out

        return output, probs


# Train without load balancing and track expert usage
d_model, d_ff = 64, 256
num_experts = 8

model_no_balance = MoELayerNoBalance(d_model, d_ff, num_experts, top_k=1)
optimizer = torch.optim.Adam(model_no_balance.parameters(), lr=1e-3)

# Simple regression task
expert_usage_history = []
for step in range(300):
    x = torch.randn(8, 32, d_model)
    target = torch.randn(8, 32, d_model)

    output, probs = model_no_balance(x)
    loss = F.mse_loss(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Track expert usage
    with torch.no_grad():
        _, top_idx = torch.topk(probs, 1, dim=-1)
        usage = torch.zeros(num_experts)
        for e in range(num_experts):
            usage[e] = (top_idx == e).float().mean().item()
        expert_usage_history.append(usage.numpy())

expert_usage_history = np.array(expert_usage_history)

# Plot expert usage over time
plt.figure(figsize=(12, 5))
for e in range(num_experts):
    plt.plot(expert_usage_history[:, e], label=f'Expert {e}', linewidth=1.5)
plt.axhline(y=1/num_experts, color='gray', linestyle='--',
            label=f'Ideal: {1/num_experts:.3f}', linewidth=2)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Fraction of Tokens', fontsize=12)
plt.title('Expert Collapse WITHOUT Load Balancing', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
#@title 🎧 What to Look For: Simulate Collapse Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_simulate_collapse_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("Final expert usage:")
for e in range(num_experts):
    pct = expert_usage_history[-1, e] * 100
    status = "COLLAPSED" if pct < 2 else ("OVERLOADED" if pct > 25 else "OK")
    print(f"  Expert {e}: {pct:.1f}% [{status}]")

In [ ]:
#@title 🎧 Listen: Aux Loss Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_aux_loss_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. The Auxiliary Load Balancing Loss

The Switch Transformer introduced an auxiliary loss that penalizes uneven expert usage:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

where:
- $f_i$ = fraction of tokens routed to expert $i$
- $P_i$ = average router probability for expert $i$
- $\alpha$ = balancing coefficient (typically 0.01)
- $N$ = number of experts

In [ ]:
#@title 🎧 Code Walkthrough: Aux Loss Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_aux_loss_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_load_balancing_loss(router_probs, top_k_indices, num_experts, alpha=0.01):
    """
    Compute the Switch Transformer auxiliary load balancing loss.

    Args:
        router_probs: (batch, seq_len, num_experts) -- softmax probabilities
        top_k_indices: (batch, seq_len, top_k) -- selected expert indices
        num_experts: int
        alpha: float -- balancing coefficient

    Returns:
        loss: scalar tensor (differentiable)
    """
    B, S, N = router_probs.shape
    K = top_k_indices.shape[-1]

    # f_i: fraction of tokens routed to each expert
    # Use one-hot encoding for differentiability
    one_hot = F.one_hot(top_k_indices, num_classes=num_experts).float()  # (B, S, K, N)
    f_i = one_hot.sum(dim=(0, 1, 2)) / (B * S * K)  # (N,)

    # P_i: average router probability for each expert
    P_i = router_probs.mean(dim=(0, 1))  # (N,)

    # Auxiliary loss
    loss = alpha * num_experts * (f_i * P_i).sum()

    return loss


# Demonstrate the loss for balanced vs imbalanced scenarios
N = 4

# Balanced case: uniform distribution
balanced_probs = torch.ones(1, 100, N) / N
balanced_indices = torch.arange(100).unsqueeze(0).unsqueeze(-1) % N  # (1, 100, 1)

balanced_loss = compute_load_balancing_loss(balanced_probs, balanced_indices, N, alpha=0.01)
print(f"Balanced case:")
print(f"  f_i = {[1/N]*N}")
print(f"  P_i = {[1/N]*N}")
print(f"  Loss = {balanced_loss.item():.6f}")

# Imbalanced case: Expert 0 gets 70% of tokens
imbalanced_probs = torch.tensor([[[0.60, 0.13, 0.13, 0.14]]] * 100).unsqueeze(0)  # (1, 100, 4)
imbalanced_indices = torch.zeros(1, 100, 1, dtype=torch.long)  # All go to expert 0
imbalanced_indices[0, 30:40, 0] = 1  # 10% to expert 1
imbalanced_indices[0, 40:50, 0] = 2  # 10% to expert 2
imbalanced_indices[0, 50:60, 0] = 3  # 10% to expert 3

imbalanced_loss = compute_load_balancing_loss(imbalanced_probs, imbalanced_indices, N, alpha=0.01)
print(f"\nImbalanced case:")
print(f"  Expert 0: 70% of tokens")
print(f"  Loss = {imbalanced_loss.item():.6f}")

print(f"\nRatio: {imbalanced_loss.item() / balanced_loss.item():.2f}x higher loss for imbalanced")

In [ ]:
#@title 🎧 Listen: Training With Balance Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_training_with_balance_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Training with Load Balancing

Now let us add the auxiliary loss to training and see how it prevents expert collapse.

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}} + \mathcal{L}_{\text{aux}}$$

In [ ]:
#@title 🎧 Code Walkthrough: Training With Balance Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_training_with_balance_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class MoELayerWithBalance(nn.Module):
    """MoE layer WITH auxiliary load balancing loss."""
    def __init__(self, d_model, d_ff, num_experts, top_k=1, alpha=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.alpha = alpha
        self.router = nn.Linear(d_model, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, S, d = x.shape
        logits = self.router(x)
        probs = F.softmax(logits, dim=-1)
        top_k_probs, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(x)
        for k in range(self.top_k):
            for e in range(self.num_experts):
                mask = (top_k_idx[:, :, k] == e)
                if mask.any():
                    tokens = x[mask]
                    expert_out = self.experts[e](tokens)
                    output[mask] += top_k_weights[:, :, k][mask].unsqueeze(-1) * expert_out

        # Compute auxiliary loss
        aux_loss = compute_load_balancing_loss(
            probs, top_k_idx, self.num_experts, self.alpha
        )

        return output, probs, aux_loss


# Train WITH load balancing
torch.manual_seed(42)
model_balanced = MoELayerWithBalance(d_model, d_ff, num_experts, top_k=1, alpha=0.01)
optimizer_b = torch.optim.Adam(model_balanced.parameters(), lr=1e-3)

usage_history_balanced = []
task_losses = []
aux_losses = []

for step in range(300):
    x = torch.randn(8, 32, d_model)
    target = torch.randn(8, 32, d_model)

    output, probs, aux_loss = model_balanced(x)
    task_loss = F.mse_loss(output, target)
    total_loss = task_loss + aux_loss

    optimizer_b.zero_grad()
    total_loss.backward()
    optimizer_b.step()

    task_losses.append(task_loss.item())
    aux_losses.append(aux_loss.item())

    with torch.no_grad():
        _, top_idx = torch.topk(probs, 1, dim=-1)
        usage = torch.zeros(num_experts)
        for e in range(num_experts):
            usage[e] = (top_idx == e).float().mean().item()
        usage_history_balanced.append(usage.numpy())

usage_history_balanced = np.array(usage_history_balanced)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Expert usage WITHOUT balancing
for e in range(num_experts):
    axes[0].plot(expert_usage_history[:, e], linewidth=1.5)
axes[0].axhline(y=1/num_experts, color='gray', linestyle='--', linewidth=2)
axes[0].set_title('WITHOUT Load Balancing', fontsize=13)
axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Token Fraction', fontsize=12)
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)

# Expert usage WITH balancing
for e in range(num_experts):
    axes[1].plot(usage_history_balanced[:, e], linewidth=1.5)
axes[1].axhline(y=1/num_experts, color='gray', linestyle='--', linewidth=2)
axes[1].set_title('WITH Load Balancing (alpha=0.01)', fontsize=13)
axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel('Token Fraction', fontsize=12)
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.3)

# Loss curves
axes[2].plot(task_losses, label='Task Loss', color='#2196F3')
axes[2].plot(aux_losses, label='Aux Loss', color='#f44336')
axes[2].set_title('Loss Components', fontsize=13)
axes[2].set_xlabel('Step', fontsize=12)
axes[2].set_ylabel('Loss', fontsize=12)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
#@title 🎧 What to Look For: Training With Balance Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_training_with_balance_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("Final usage (with balancing):")
for e in range(num_experts):
    pct = usage_history_balanced[-1, e] * 100
    print(f"  Expert {e}: {pct:.1f}%")
print(f"\nIdeal per expert: {100/num_experts:.1f}%")

In [ ]:
#@title 🎧 Listen: Capacity Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_capacity_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Expert Capacity and Token Dropping

Even with the auxiliary loss, perfect balance is never achieved. We set a **capacity factor** $C$ to limit how many tokens each expert can process:

$$\text{Capacity}_i = C \cdot \frac{T \cdot k}{N}$$

Tokens that exceed capacity are **dropped** -- they pass through via the residual connection.

In [ ]:
#@title 🎧 Code Walkthrough: Capacity Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_capacity_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class MoELayerWithCapacity(nn.Module):
    """MoE layer with capacity-based token dropping."""
    def __init__(self, d_model, d_ff, num_experts, top_k=1,
                 alpha=0.01, capacity_factor=1.25):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.alpha = alpha
        self.capacity_factor = capacity_factor
        self.router = nn.Linear(d_model, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, S, d = x.shape
        T = B * S

        # Router
        logits = self.router(x)
        probs = F.softmax(logits, dim=-1)
        top_k_probs, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # Compute capacity
        ideal_per_expert = T * self.top_k / self.num_experts
        capacity = int(self.capacity_factor * ideal_per_expert)

        # Track statistics
        tokens_processed = 0
        tokens_dropped = 0

        output = torch.zeros_like(x)
        expert_counts = torch.zeros(self.num_experts)

        for k in range(self.top_k):
            for e in range(self.num_experts):
                mask = (top_k_idx[:, :, k] == e)
                if not mask.any():
                    continue

                current_count = expert_counts[e].item()
                num_tokens = mask.sum().item()

                if current_count >= capacity:
                    # All tokens dropped
                    tokens_dropped += num_tokens
                    continue

                # Check how many we can still accept
                remaining_capacity = capacity - int(current_count)

                if num_tokens <= remaining_capacity:
                    # Process all tokens
                    tokens = x[mask]
                    expert_out = self.experts[e](tokens)
                    output[mask] += top_k_weights[:, :, k][mask].unsqueeze(-1) * expert_out
                    expert_counts[e] += num_tokens
                    tokens_processed += num_tokens
                else:
                    # Process only up to capacity
                    flat_mask = mask.view(-1)
                    token_positions = flat_mask.nonzero(as_tuple=True)[0]
                    keep = token_positions[:remaining_capacity]

                    keep_mask = torch.zeros_like(flat_mask)
                    keep_mask[keep] = True
                    keep_mask = keep_mask.view(B, S)

                    tokens = x[keep_mask]
                    expert_out = self.experts[e](tokens)
                    output[keep_mask] += top_k_weights[:, :, k][keep_mask].unsqueeze(-1) * expert_out

                    expert_counts[e] += remaining_capacity
                    tokens_processed += remaining_capacity
                    tokens_dropped += (num_tokens - remaining_capacity)

        # Auxiliary loss
        aux_loss = compute_load_balancing_loss(
            probs, top_k_idx, self.num_experts, self.alpha
        )

        stats = {
            'tokens_processed': tokens_processed,
            'tokens_dropped': tokens_dropped,
            'drop_rate': tokens_dropped / (tokens_processed + tokens_dropped) if (tokens_processed + tokens_dropped) > 0 else 0,
            'expert_counts': expert_counts,
            'capacity': capacity,
        }

        return output, probs, aux_loss, stats


def compute_load_balancing_loss(router_probs, top_k_indices, num_experts, alpha=0.01):
    B, S, N = router_probs.shape
    K = top_k_indices.shape[-1]
    one_hot = F.one_hot(top_k_indices, num_classes=num_experts).float()
    f_i = one_hot.sum(dim=(0, 1, 2)) / (B * S * K)
    P_i = router_probs.mean(dim=(0, 1))
    return alpha * num_experts * (f_i * P_i).sum()


# Test capacity dropping
model_cap = MoELayerWithCapacity(
    d_model, d_ff, num_experts, top_k=1,
    alpha=0.01, capacity_factor=1.25
)

x = torch.randn(4, 128, d_model)  # 512 tokens
out, probs, aux, stats = model_cap(x)

print(f"Capacity per expert: {stats['capacity']}")
print(f"Ideal per expert: {512 * 1 / num_experts:.0f}")
print(f"Tokens processed: {stats['tokens_processed']}")
print(f"Tokens dropped: {stats['tokens_dropped']}")
print(f"Drop rate: {stats['drop_rate']*100:.1f}%")
print(f"\nExpert token counts:")
for e in range(num_experts):
    count = int(stats['expert_counts'][e].item())
    bar = '#' * (count // 2)
    print(f"  Expert {e}: {count:>4} {bar}")

In [ ]:
#@title 🎧 Narration: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn -- TODO Exercises

### TODO 1: Explore the Effect of Alpha

Train the MoE layer with different alpha values and measure the trade-off between load balance and task performance.

In [ ]:
#@title 🎧 Before You Start: Todo1 Before
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_todo1_before.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Train with alpha in [0, 0.001, 0.01, 0.1, 1.0]
# For each, record:
#   - Final task loss
#   - Final expert usage std deviation (imbalance measure)
#   - Number of "collapsed" experts (usage < 2%)

alphas = [0, 0.001, 0.01, 0.1, 1.0]
results = []

for alpha in alphas:
    torch.manual_seed(42)
    model = MoELayerWithBalance(d_model, d_ff, num_experts, top_k=1, alpha=alpha)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for step in range(200):
        x = torch.randn(8, 32, d_model)
        target = torch.randn(8, 32, d_model)

        output, probs, aux_loss = model(x)
        task_loss = F.mse_loss(output, target)
        total_loss = task_loss + aux_loss

        opt.zero_grad()
        total_loss.backward()
        opt.step()

    # Evaluate final state
    with torch.no_grad():
        x_eval = torch.randn(16, 64, d_model)
        _, eval_probs, _ = model(x_eval)
        _, top_idx = torch.topk(eval_probs, 1, dim=-1)
        usage = torch.zeros(num_experts)
        for e in range(num_experts):
            usage[e] = (top_idx == e).float().mean().item()

    # YOUR CODE HERE: compute metrics
    final_task_loss = task_loss.item()
    usage_std = ???  # Standard deviation of expert usage
    collapsed = ???  # Number of experts with usage < 0.02

    results.append({
        'alpha': alpha,
        'task_loss': final_task_loss,
        'usage_std': usage_std,
        'collapsed': collapsed,
        'usage': usage.numpy(),
    })
# ==============================


In [ ]:
#@title 🎧 Narration: Todo1 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_todo1_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Display results
print(f"{'Alpha':>8} {'Task Loss':>12} {'Usage Std':>12} {'Collapsed':>12}")
print("-" * 48)
for r in results:
    print(f"{r['alpha']:>8.3f} {r['task_loss']:>12.4f} {r['usage_std']:>12.4f} {r['collapsed']:>12d}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([r['alpha'] for r in results], [r['task_loss'] for r in results],
             'o-', color='#2196F3', linewidth=2, markersize=8)
axes[0].set_xlabel('Alpha (Balancing Coefficient)', fontsize=12)
axes[0].set_ylabel('Task Loss', fontsize=12)
axes[0].set_title('Task Performance vs Alpha', fontsize=13)
axes[0].set_xscale('symlog', linthresh=0.0001)
axes[0].grid(True, alpha=0.3)

axes[1].plot([r['alpha'] for r in results], [r['usage_std'] for r in results],
             'o-', color='#f44336', linewidth=2, markersize=8)
axes[1].set_xlabel('Alpha (Balancing Coefficient)', fontsize=12)
axes[1].set_ylabel('Expert Usage Std Dev (lower = more balanced)', fontsize=12)
axes[1].set_title('Load Balance vs Alpha', fontsize=13)
axes[1].set_xscale('symlog', linthresh=0.0001)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSweet spot: alpha=0.01 typically gives good balance without hurting task loss.")

In [ ]:
#@title 🎧 Before You Start: Todo2 Before
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_todo2_before.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Explore Capacity Factor Trade-Offs

Vary the capacity factor and measure how many tokens get dropped vs how balanced the load is.

In [ ]:
# ============ TODO ============
# Test capacity factors: [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
# For each, measure:
#   - Drop rate (fraction of tokens dropped)
#   - Max expert load / min expert load ratio
#   - Memory usage (capacity * num_experts * d_model * dtype_bytes)

capacity_factors = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
cap_results = []

for cf in capacity_factors:
    torch.manual_seed(42)
    model = MoELayerWithCapacity(
        d_model, d_ff, num_experts, top_k=1,
        alpha=0.01, capacity_factor=cf
    )

    # Evaluate on a batch
    x = torch.randn(8, 64, d_model)  # 512 tokens
    with torch.no_grad():
        _, _, _, stats = model(x)

    # YOUR CODE HERE: compute memory for expert buffers
    buffer_memory_MB = ???  # stats['capacity'] * num_experts * d_model * 2 / 1e6

    cap_results.append({
        'cf': cf,
        'drop_rate': stats['drop_rate'],
        'capacity': stats['capacity'],
        'buffer_MB': buffer_memory_MB,
        'max_load': int(stats['expert_counts'].max().item()),
        'min_load': int(stats['expert_counts'].min().item()),
    })
# ==============================


In [ ]:
#@title 🎧 Narration: Todo2 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_todo2_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"{'C':>6} {'Capacity':>10} {'Drop Rate':>10} {'Max Load':>10} {'Min Load':>10} {'Buffer MB':>10}")
print("-" * 58)
for r in cap_results:
    print(f"{r['cf']:>6.2f} {r['capacity']:>10} {r['drop_rate']*100:>9.1f}% "
          f"{r['max_load']:>10} {r['min_load']:>10} {r['buffer_MB']:>10.2f}")

# Plot
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot([r['cf'] for r in cap_results],
         [r['drop_rate']*100 for r in cap_results],
         'o-', color='#f44336', linewidth=2, markersize=8, label='Drop Rate (%)')
ax1.set_xlabel('Capacity Factor', fontsize=12)
ax1.set_ylabel('Token Drop Rate (%)', color='#f44336', fontsize=12)

ax2 = ax1.twinx()
ax2.plot([r['cf'] for r in cap_results],
         [r['buffer_MB'] for r in cap_results],
         's-', color='#2196F3', linewidth=2, markersize=8, label='Buffer Memory (MB)')
ax2.set_ylabel('Buffer Memory (MB)', color='#2196F3', fontsize=12)

plt.title('Capacity Factor Trade-Off: Drop Rate vs Memory', fontsize=14)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc='center right')
plt.tight_layout()
plt.show()

print("\nSweet spot: C=1.1 to C=1.5 balances drop rate and memory.")

In [ ]:
#@title 🎧 Listen: 5d Grid Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_5d_grid_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. The 5D Parallelism Grid

Expert parallelism is the fifth and final dimension of the parallelism grid. Let us build a calculator that determines the configuration for a given number of GPUs.

$$\text{Total GPUs} = \text{DP} \times \text{TP} \times \text{PP} \times \text{SP/CP} \times \text{EP}$$

In [ ]:
#@title 🎧 Code Walkthrough: 5d Grid Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_5d_grid_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class ParallelismConfig:
    """5D parallelism configuration calculator."""

    def __init__(self, dp, tp, pp, sp_cp, ep):
        self.dp = dp        # Data Parallelism
        self.tp = tp        # Tensor Parallelism
        self.pp = pp        # Pipeline Parallelism
        self.sp_cp = sp_cp  # Sequence/Context Parallelism
        self.ep = ep        # Expert Parallelism

    @property
    def total_gpus(self):
        return self.dp * self.tp * self.pp * self.sp_cp * self.ep

    def describe(self, name=""):
        print(f"{'=' * 60}")
        if name:
            print(f"  {name}")
        print(f"  Total GPUs: {self.total_gpus}")
        print(f"  DP={self.dp} x TP={self.tp} x PP={self.pp} "
              f"x SP/CP={self.sp_cp} x EP={self.ep}")
        print(f"{'=' * 60}")

        dims = [
            ("Data Parallelism (DP)", self.dp,
             "Splits batches", "AllReduce (gradients)"),
            ("Tensor Parallelism (TP)", self.tp,
             "Splits weight matrices", "AllReduce (per layer)"),
            ("Pipeline Parallelism (PP)", self.pp,
             "Splits transformer layers", "Point-to-point (activations)"),
            ("Sequence/Context (SP/CP)", self.sp_cp,
             "Splits sequence dimension", "Ring Attention / AllGather"),
            ("Expert Parallelism (EP)", self.ep,
             "Splits MoE experts", "All-to-All"),
        ]

        print(f"\n  {'Dimension':<28} {'Size':>6} {'What is Split':<25} {'Communication'}")
        print(f"  {'-'*90}")
        for name, size, split, comm in dims:
            active = '*' if size > 1 else ' '
            print(f"  {active} {name:<26} {size:>6} {split:<25} {comm}")


# Real-world configurations
configs = [
    ("DeepSeek-V3 (2048 H800 GPUs)",
     ParallelismConfig(dp=2, tp=1, pp=16, sp_cp=1, ep=64)),
    ("Megatron-LM Dense (256 A100 GPUs)",
     ParallelismConfig(dp=16, tp=8, pp=2, sp_cp=1, ep=1)),
    ("Switch Transformer (2048 TPU cores)",
     ParallelismConfig(dp=16, tp=1, pp=1, sp_cp=1, ep=128)),
    ("Mixtral-like (64 GPUs)",
     ParallelismConfig(dp=4, tp=2, pp=1, sp_cp=1, ep=8)),
]

for name, config in configs:
    config.describe(name)
    print()

In [ ]:
#@title 🎧 Before You Start: Todo3 Before
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_todo3_before.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Design Your Own 5D Configuration

Given a set of constraints, design a parallelism configuration.

In [ ]:
# ============ TODO ============
# You have 512 GPUs. Your model has:
#   - 128 experts (needs EP >= 16)
#   - 80 transformer layers (needs PP >= 4)
#   - Large hidden dim 8192 (needs TP >= 4)
#   - Long sequences 128K (might need SP/CP >= 2)
#
# Constraints:
#   - TP must be <= 8 (within one node)
#   - EP * TP should use fast interconnect
#   - DP * TP * PP * SP/CP * EP = 512
#
# Design your configuration:
my_config = ParallelismConfig(
    dp=???,      # YOUR CODE HERE
    tp=???,      # YOUR CODE HERE
    pp=???,      # YOUR CODE HERE
    sp_cp=???,   # YOUR CODE HERE
    ep=???,      # YOUR CODE HERE
)
# ==============================


In [ ]:
#@title 🎧 Narration: Todo3 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_todo3_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
assert my_config.total_gpus == 512, \
    f"Must use exactly 512 GPUs! Got {my_config.total_gpus}"
assert my_config.ep >= 16, "Need at least EP=16 for 128 experts"
assert my_config.pp >= 4, "Need at least PP=4 for 80 layers"
assert my_config.tp <= 8, "TP must be <= 8 (within one node)"

my_config.describe("My 512-GPU Configuration")

experts_per_ep_gpu = 128 // my_config.ep
layers_per_pp_stage = 80 // my_config.pp
print(f"\nDerived:")
print(f"  Experts per EP-GPU: {experts_per_ep_gpu}")
print(f"  Layers per PP-stage: {layers_per_pp_stage}")
print(f"  Data-parallel replicas: {my_config.dp}")
print(f"\nTODO 3 passed!")

In [ ]:
#@title 🎧 Transition: Course Summary Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_course_summary_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Complete Course Summary

Let us review what each pod in the GPU Parallelism course covered and how the strategies compose.

In [ ]:
#@title 🎧 What to Look For: Course Summary Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_course_summary_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Course summary visualization
pods = [
    ("Pod 1: GPU Fundamentals", "Compute", "Thousands of cores for matrix multiplications"),
    ("Pod 2: Memory Components", "Memory", "Weights + Gradients + Optimizer + Activations"),
    ("Pod 3: Recomp + Accum + DP", "Memory/Compute", "Trade compute for memory, replicate model"),
    ("Pod 4: AllReduce + Profiling", "Communication", "Efficient gradient sync, batch size scaling"),
    ("Pod 5: ZeRO Optimizer", "Memory", "Partition optimizer/gradients/params across GPUs"),
    ("Pod 6: Tensor Parallelism", "Memory", "Split weight matrices (column/row sharding)"),
    ("Pod 7: Sequence Parallelism", "Memory", "Split LayerNorm/dropout sequence dim"),
    ("Pod 8: Context Parallelism", "Seq Length", "Ring Attention for million-token contexts"),
    ("Pod 9: Pipeline Parallelism", "Memory/Compute", "Split layers across GPUs, 1F1B schedules"),
    ("Pod 10: Expert Parallelism", "Capacity", "Distribute MoE experts, All-to-All, load balance"),
]

bottleneck_colors = {
    "Compute": "#2196F3",
    "Memory": "#4CAF50",
    "Memory/Compute": "#8BC34A",
    "Communication": "#FF9800",
    "Seq Length": "#9C27B0",
    "Capacity": "#f44336",
}

fig, ax = plt.subplots(figsize=(14, 8))

for i, (name, bottleneck, desc) in enumerate(reversed(pods)):
    y = i
    color = bottleneck_colors[bottleneck]
    ax.barh(y, 1, color=color, alpha=0.8, height=0.7, edgecolor='white')
    ax.text(0.02, y, f"{name}", va='center', fontsize=11, fontweight='bold',
            color='white')
    ax.text(1.05, y, f"[{bottleneck}] {desc}", va='center', fontsize=10)

ax.set_xlim(-0.1, 4.5)
ax.set_yticks([])
ax.set_title('GPU Parallelism Course: 10-Pod Roadmap', fontsize=16, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_xticks([])

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=b) for b, c in bottleneck_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10,
          title='Bottleneck Addressed')

plt.tight_layout()
plt.show()

print("\nThe 5D Parallelism Formula:")
print("  Total GPUs = DP x TP x PP x SP/CP x EP")
print("\nEach dimension addresses a different bottleneck.")
print("No single strategy is sufficient at the largest scales.")
print("The power comes from their composition.")

In [ ]:
#@title 🎧 Listen: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Reflection and Key Takeaways

### What We Built in This Notebook
- **Auxiliary load balancing loss**: $\mathcal{L}_{\text{aux}} = \alpha \cdot N \cdot \sum f_i \cdot P_i$
- **Capacity-based token dropping**: prevents GPU bottlenecks at the cost of dropped tokens
- **5D parallelism configuration**: calculator for real-world deployments

### Key Takeaways from the Full Course
1. **Load balancing is critical** -- without it, MoE models collapse to using 1-2 experts
2. **Alpha = 0.01** is a good starting point -- it balances utilization without hurting quality
3. **Capacity factor C = 1.1 to 1.5** handles most imbalance without excessive memory
4. **The 5D grid** composes five independent strategies, each addressing a different bottleneck
5. **Real systems like DeepSeek-V3** use all five dimensions simultaneously on 2048+ GPUs

### What You Now Understand
When you read about a model trained on 10,000 GPUs, you can reason about:
- How the GPUs are organized (5D grid)
- What each GPU computes (its slice of the model/data)
- How they communicate (AllReduce, All-to-All, point-to-point)
- Why load balance matters (expert collapse, GPU idle time)


In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Congratulations on completing all 10 pods of the GPU Parallelism course!